In [1]:
import os
from pyspark.sql import SparkSession

# ENRICHED FIX: Adding official PostgreSQL JDBC Driver jar package coordinates to Spark Submit environment
# This allows the distributed compute cluster to communicate seamlessly with our OLTP layer
SUBMIT_PACKAGES = (
    "io.delta:delta-spark_2.12:3.0.0,"
    "org.apache.hadoop:hadoop-aws:3.3.4,"
    "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0,"
    "org.postgresql:postgresql:42.6.0"
)

os.environ["PYSPARK_SUBMIT_ARGS"] = f"--packages {SUBMIT_PACKAGES} pyspark-shell"

# Initialize Spark Session with clean configurations and HDFSLogStore
spark = SparkSession.builder \
    .appName("Ecommerce_Data_Lakehouse_Pipeline") \
    .master("local[*]") \
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension") \
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog") \
    .config("spark.delta.logStore.class", "org.apache.spark.sql.delta.storage.HDFSLogStore") \
    .getOrCreate()

# Synchronize global Java Hadoop Configuration variables
hadoop_conf = spark._jsc.hadoopConfiguration()
hadoop_conf.set("fs.s3a.endpoint", "http://lh_minio:9000")
hadoop_conf.set("fs.s3a.access.key", "admin")
hadoop_conf.set("fs.s3a.secret.key", "minio_password123")
hadoop_conf.set("fs.s3a.path.style.access", "true")
hadoop_conf.set("fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
hadoop_conf.set("fs.s3a.connection.ssl.enabled", "false")

print("🚀 Hardened PySpark Session with Integrated PostgreSQL JDBC Driver Driver Active!")


🚀 Hardened PySpark Session with Integrated PostgreSQL JDBC Driver Driver Active!


In [2]:
from pyspark.sql.functions import col, current_timestamp

# Defining clean local file system targets inside the notebook environment workspace
# This eliminates network URI parsing bottlenecks while preserving 100% Delta Lake ACID properties
BRONZE_PATH = "file:///home/jovyan/work/notebooks/data/bronze/clickstream"
CHECKPOINT_PATH_BRONZE = "file:///home/jovyan/work/notebooks/data/checkpoints/bronze_clickstream"

# Establish a continuous live connection string targeting the local Apache Kafka broker cluster
kafka_raw_stream = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:29092") \
    .option("subscribe", "clickstream_events") \
    .option("startingOffsets", "latest") \
    .load()

# Metadata Ingestion Layer: Extract binary network payloads into structured metadata attributes
bronze_stream_df = kafka_raw_stream.select(
    col("value").cast("string").alias("raw_payload"),
    col("topic").alias("kafka_topic"),
    col("partition").alias("kafka_partition"),
    col("offset").alias("kafka_offset"),
    current_timestamp().alias("ingested_at")
)

# Start execution with transaction logs handled perfectly via the local filesystem kernel
bronze_query = bronze_stream_df.writeStream \
    .format("delta") \
    .option("checkpointLocation", CHECKPOINT_PATH_BRONZE) \
    .outputMode("append") \
    .start(BRONZE_PATH)

print("🥉 Real-Time Ingestion Pipeline Secured: Listening to Kafka and actively appending rows to Bronze Delta Lake!")


🥉 Real-Time Ingestion Pipeline Secured: Listening to Kafka and actively appending rows to Bronze Delta Lake!


In [3]:
from pyspark.sql.functions import from_json, col
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType

SILVER_PATH = "file:///home/jovyan/work/notebooks/data/silver/clickstream"
CHECKPOINT_PATH_SILVER = "file:///home/jovyan/work/notebooks/data/checkpoints/silver_clickstream"

clickstream_schema = StructType([
    StructField("user_id", IntegerType(), True),
    StructField("page_url", StringType(), True),
    StructField("ip_address", StringType(), True),
    StructField("device_type", StringType(), True),
    StructField("click_timestamp", StringType(), True)
])

# Stream reader will now automatically infer the correct schema (including LongType offsets!) from disk logs
bronze_source_stream = spark.readStream \
    .format("delta") \
    .load("file:///home/jovyan/work/notebooks/data/bronze/clickstream")

parsed_silver_df = bronze_source_stream \
    .select(from_json(col("raw_payload"), clickstream_schema).alias("data"), col("ingested_at")) \
    .select("data.*", "ingested_at") \
    .withColumn("click_timestamp", col("click_timestamp").cast(TimestampType())) \
    .filter(col("user_id").isNotNull())

silver_query = parsed_silver_df.writeStream \
    .format("delta") \
    .option("checkpointLocation", CHECKPOINT_PATH_SILVER) \
    .outputMode("append") \
    .start(SILVER_PATH)

print("🥈 Silver Layer Pipeline Activated: Continuous parsing active over auto-inferred structure!")


🥈 Silver Layer Pipeline Activated: Continuous parsing active over auto-inferred structure!


In [4]:
from pyspark.sql.functions import desc

GOLD_PATH = "file:///home/jovyan/work/notebooks/data/gold/page_click_aggregates"
CHECKPOINT_PATH_GOLD = "file:///home/jovyan/work/notebooks/data/checkpoints/gold_click_aggregates"

silver_source_stream = spark.readStream \
    .format("delta") \
    .load("file:///home/jovyan/work/notebooks/data/silver/clickstream")

gold_aggregated_df = silver_source_stream \
    .groupBy("page_url") \
    .count() \
    .withColumnRenamed("count", "total_clicks")

gold_query = gold_aggregated_df.writeStream \
    .format("delta") \
    .trigger(processingTime="2 seconds") \
    .option("checkpointLocation", CHECKPOINT_PATH_GOLD) \
    .outputMode("complete") \
    .start(GOLD_PATH)

print("🥇 Gold Layer Pipeline Activated with Strict Trigger Intervals!")


🥇 Gold Layer Pipeline Activated with Strict Trigger Intervals!


In [7]:
import time
from pyspark.sql.functions import count

# Paths for transactional evaluation
BRONZE_SNAP_PATH = "file:///home/jovyan/work/notebooks/data/bronze/clickstream"
SILVER_SNAP_PATH = "file:///home/jovyan/work/notebooks/data/silver/clickstream"

print("📊 Fetching Data Quality Governance Telemetry Metrics...")

# 1. Read frozen snapshot profiles of active Delta targets
df_bronze_snapshot = spark.read.format("delta").load(BRONZE_SNAP_PATH)
df_silver_snapshot = spark.read.format("delta").load(SILVER_SNAP_PATH)

total_bronze_rows = df_bronze_snapshot.count()
total_silver_rows = df_silver_snapshot.count()

# Calculate metadata metrics regarding data quality drop execution rates
dropped_rows_count = total_bronze_rows - total_silver_rows
data_cleanliness_efficiency = (total_silver_rows / total_bronze_rows) * 100 if total_bronze_rows > 0 else 100.0

print("\n==============================================================")
print("📈 LAKHOUSE TELEMETRY & DATA CLEANING ENGINE REPORT")
print("==============================================================")
print(f"🔹 Total Raw Records Ingested (Bronze Layer): {total_bronze_rows} rows")
print(f"🔹 Total Cleaned Records Retained (Silver Layer): {total_silver_rows} rows")
print(f"🔸 Total Anomalous Rows Dropped via Data Quality Fence: {dropped_rows_count} rows")
print(f"📊 Data Ingestion Cleanliness Efficiency Score: {round(data_cleanliness_efficiency, 2)}%")
print("==============================================================")


📊 Fetching Data Quality Governance Telemetry Metrics...

📈 LAKHOUSE TELEMETRY & DATA CLEANING ENGINE REPORT
🔹 Total Raw Records Ingested (Bronze Layer): 2481 rows
🔹 Total Cleaned Records Retained (Silver Layer): 2428 rows
🔸 Total Anomalous Rows Dropped via Data Quality Fence: 53 rows
📊 Data Ingestion Cleanliness Efficiency Score: 97.86%


In [6]:
# CRITICAL NETWORK FIX: Changing from 'localhost' to 'host.docker.internal' 
# This instructs the Spark container to route its database traffic through the host OS loopback bridge to hit port 5433
pg_users_df = spark.read \
    .format("jdbc") \
    .option("url", "jdbc:postgresql://host.docker.internal:5433/ecommerce_db") \
    .option("dbtable", "users") \
    .option("user", "data_engineer") \
    .option("password", "de_password123") \
    .option("driver", "org.postgresql.Driver") \
    .load()

# Read the clean streaming traces from the local Delta Lake space
silver_clicks_df = spark.read \
    .format("delta") \
    .load("file:///home/jovyan/work/notebooks/data/silver/clickstream")

print("🔄 Executing Cross-Platform Distributed Join Operation (OLTP Core + OLAP Telemetry Stream)...")

# Perform high-speed join combining relational database profiles with live telemetry logs
unified_polyglot_insights = silver_clicks_df \
    .join(pg_users_df, silver_clicks_df.user_id == pg_users_df.user_id, "inner") \
    .select(
        pg_users_df.user_id,
        pg_users_df.first_name,
        pg_users_df.email,
        silver_clicks_df.page_url,
        silver_clicks_df.device_type,
        silver_clicks_df.click_timestamp
    )

# Display the final unified analytical matrix snapshot
unified_polyglot_insights.orderBy(silver_clicks_df.click_timestamp.desc()).show(20, truncate=False)


🔄 Executing Cross-Platform Distributed Join Operation (OLTP Core + OLAP Telemetry Stream)...
+-------+-----------+------------------------------------+---------------------+--------------+-------------------+
|user_id|first_name |email                               |page_url             |device_type   |click_timestamp    |
+-------+-----------+------------------------------------+---------------------+--------------+-------------------+
|22135  |Rebecca    |rebecca.kirby_797d4@gmail.com       |https://ecommerce.com|mobile_android|2026-05-18 19:57:17|
|23916  |Cynthia    |cynthia.johnson_3bdf8@gmail.com     |https://ecommerce.com|mobile_ios    |2026-05-18 19:57:17|
|24343  |Jessica    |jessica.silva_d6ab8@hotmail.com     |https://ecommerce.com|mobile_ios    |2026-05-18 19:57:16|
|47710  |Jeremy     |jeremy.johnson_e337d@hotmail.com    |https://ecommerce.com|desktop       |2026-05-18 19:57:16|
|29773  |Gerald     |gerald.douglas_cba48@outlook.com    |https://ecommerce.com|mobile_android|